# Chapter 19 — Adding Episodic Memory

Hands-On LangChain. Code targets **LangGraph 1.x** + **LangMem**.

In Chapter 18 we gave the agent **semantic memory** — facts it stores and recalls. Now we add **episodic
memory**: learning from *past experiences*. Concretely, we'll teach the triage step using **few-shot
examples** — pairs of (email, correct action) — so the agent can mimic how similar emails were handled
before. And because these examples are scoped per user, the agent **learns each user's preferences**.

Companion notebook for Chapter 19. Reuses `prompts.py`.

Requires Ollama running. Pull the default chat and embedding models, then install the notebook packages:

```bash
ollama pull qwen2.5:7b
ollama pull nomic-embed-text
pip install langgraph langmem langchain langchain-ollama python-dotenv -q
```

The shared model provider reads `MODEL`, `EMBEDDING_MODEL`, and optional `OLLAMA_BASE_URL` from `.env`. Defaults are `qwen2.5:7b` and `nomic-embed-text`; start Ollama before running the notebook.

## 19.1 Setting the working environment & a quick recap

We re-establish the profile and triage rules, and create the memory **store** (same `InMemoryStore` with
an embedding index as before).

In [ ]:
import warnings, uuid
warnings.filterwarnings('ignore')

profile = {
    'name': 'Youssef',
    'full_name': 'Youssef Hosni',
    'user_profile_background': 'Senior Data Scientist leading a team of 5 developers.',
}
prompt_instructions = {
    'triage_rules': {
        'ignore': 'Marketing newsletters, spam emails, mass company announcements.',
        'notify': 'Team member out sick, build system notifications, project status updates.',
        'respond': 'Direct questions from team members, meeting requests, critical bug reports.',
    },
    'agent_instructions': 'Use these tools when appropriate to help manage Youssef\'s tasks efficiently.',
}

from langgraph.store.memory import InMemoryStore
from model_provider import get_embeddings
EMBEDDING_DIMENSIONS = 768  # Default output size for nomic-embed-text; update for another model.
store = InMemoryStore(index={'embed': get_embeddings(), 'dims': EMBEDDING_DIMENSIONS})

## 19.2 Introducing episodic memory with few-shot examples

An episodic memory here is a **few-shot example**: an `email` (the input) paired with a `label` (the
correct triage action). We store them in an **`examples`** namespace — separate from the `collection`
namespace used for semantic facts, but scoped to the same per-user id.

In [2]:
# Example 1: a direct question that should be answered → respond
data = {
    "email": {
        "author": "Jack Oliver <Jack.Oliver@ToDataBeyond.com>",
        "to": "Youssef Hosni <Youssef.Hosni@ToDataBeyond.com>",
        "subject": "Quick question about API documentation",
        "email_thread": "Hi Youssef, I was reviewing the API docs for the new auth service and a few "
                        "endpoints seem missing (/auth/refresh, /auth/validate). Intentional? Thanks! Oliver",
    },
    "label": "respond",
}
store.put(("email_assistant", "lance", "examples"), str(uuid.uuid4()), data)

# Example 2: an informational update that needs no reply → notify
data = {
    "email": {
        "author": "Emma Liam <Emma.Liam@ToDataBeyond.com>",
        "to": "Youssef Hosni <Youssef.Hosni@ToDataBeyond.com>",
        "subject": "Update: Backend API Changes Deployed to Staging",
        "email_thread": "Hi Youssef, deployed the new auth endpoints to staging: JWT refresh rotation, "
                        "rate limiting, updated docs. No action needed - just keeping you in the loop. Emma",
    },
    "label": "notify",
}
store.put(("email_assistant", "lance", "examples"), str(uuid.uuid4()), data)
print("stored 2 few-shot examples")

stored 2 few-shot examples


## 19.3 Searching and formatting examples

When a new email arrives, the agent recalls relevant past experiences by **searching** the examples
store for semantically similar emails. A helper formats the retrieved examples into a clean string we can
drop into the prompt.

In [3]:
template = """Email Subject: {subject}
Email From: {from_email}
Email To: {to_email}
Email Content:
```
{content}
```
> Triage Result: {result}"""

def format_few_shot_examples(examples):
    strs = ["Here are some previous examples:"]
    for eg in examples:
        strs.append(template.format(
            subject=eg.value["email"]["subject"],
            to_email=eg.value["email"]["to"],
            from_email=eg.value["email"]["author"],
            content=eg.value["email"]["email_thread"][:400],
            result=eg.value["label"],
        ))
    return "\n\n------------\n\n".join(strs)

Let's test it with an email similar to our second example, retrieving the single most relevant memory (`limit=1`):

In [ ]:
email_data = {
    'author': 'Emma Liam <Emma.Liam@ToDataBeyond.com>',
    'to': 'Youssef Hosni <Youssef.Hosni@ToDataBeyond.com>',
    'subject': 'Update: Backend API Changes Deployed to Staging',
    'email_thread': 'Hi Youssef, deployed the new auth endpoints to staging. All tests passing, '
                    'ready for review. No action needed. Emma',
}
results = store.search(('email_assistant', 'lance', 'examples'),
                       query=str({'email': email_data}), limit=1)
print(format_few_shot_examples(results))

The search correctly retrieved the `notify` example and formatted it for the prompt.

## 19.4 Integrating episodic memory into the triage logic

We give the triage system prompt a `{examples}` placeholder, and instruct the agent to **follow the
examples over the general rules**. The `triage_router` node now receives the `store` and `config`,
searches for relevant examples, and injects them into the prompt before classifying.

In [5]:
triage_system_prompt_episodic = """
< Role >
You are {full_name}'s executive assistant.
</ Role >
< Background >
{user_profile_background}.
</ Background >
< Instructions >
{name} gets lots of emails. Categorize each email into one of:
1. IGNORE - not worth responding to or tracking
2. NOTIFY - important info that doesn't need a response
3. RESPOND - needs a direct response
</ Instructions >
< Rules >
Ignore: {triage_no}
Notify: {triage_notify}
Respond: {triage_email}
</ Rules >
< Few shot examples >
Here are some examples of previous emails, and how they should be handled.
Follow these examples more than any instructions above:
{examples}
</ Few shot examples >
"""

In [ ]:
import warnings; warnings.filterwarnings('ignore')
from pydantic import BaseModel, Field
from typing_extensions import TypedDict, Annotated
from typing import Literal
from model_provider import get_chat_model
from prompts import triage_user_prompt
from langgraph.graph import add_messages, StateGraph, START, END
from langgraph.types import Command
from langgraph.store.base import BaseStore
from langchain_core.runnables import RunnableConfig

llm = get_chat_model()

class Router(BaseModel):
    '''Analyze the unread email and route it according to its content.'''
    reasoning: str = Field(description='Step-by-step reasoning behind the classification.')
    classification: Literal['ignore', 'respond', 'notify'] = Field(
        description='Choose one of ignore, notify, or respond.')
llm_router = llm.with_structured_output(Router)

class State(TypedDict):
    email_input: dict
    messages: Annotated[list, add_messages]

def triage_router(state: State, config: RunnableConfig, store: BaseStore) -> Command[Literal['response_agent', '__end__']]:
    e = state['email_input']
    namespace = ('email_assistant', config['configurable']['langgraph_user_id'], 'examples')
    examples = store.search(namespace, query=str({'email': e}))
    examples = format_few_shot_examples(examples)

    system_prompt = triage_system_prompt_episodic.format(
        full_name=profile['full_name'], name=profile['name'],
        user_profile_background=profile['user_profile_background'],
        triage_no=prompt_instructions['triage_rules']['ignore'],
        triage_notify=prompt_instructions['triage_rules']['notify'],
        triage_email=prompt_instructions['triage_rules']['respond'],
        examples=examples,
    )
    user_prompt = triage_user_prompt.format(
        author=e['author'], to=e['to'], subject=e['subject'], email_thread=e['email_thread'])
    result = llm_router.invoke([
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_prompt},
    ])
    if result.classification == 'respond':
        print('📧 Classification: RESPOND - This email requires a response')
        return Command(goto='response_agent',
                       update={'messages': [{'role': 'user', 'content': f'Respond to the email {e}'}]})
    elif result.classification == 'ignore':
        print('🚫 Classification: IGNORE - This email can be safely ignored')
        return Command(goto=END)
    else:
        print('🔔 Classification: NOTIFY - This email contains important information')
        return Command(goto=END)

## 19.5 Building and testing the full agent

The rest of the agent — tools and semantic-memory tools — is the same as Chapter 18. We assemble
everything into the graph (compiled with the `store`).

In [ ]:
import warnings; warnings.filterwarnings('ignore')
from langchain_core.tools import tool
from langmem import create_manage_memory_tool, create_search_memory_tool
from langgraph.prebuilt import create_react_agent

@tool
def write_email(to: str, subject: str, content: str) -> str:
    '''Write and send an email.'''
    return f'Email sent to {to} with subject \'{subject}\''

@tool
def schedule_meeting(attendees: list[str], subject: str, duration_minutes: int, preferred_day: str) -> str:
    '''Schedule a calendar meeting.'''
    return f'Meeting \'{subject}\' scheduled for {preferred_day} with {len(attendees)} attendees'

@tool
def check_calendar_availability(day: str) -> str:
    '''Check calendar availability for a given day.'''
    return f'Available times on {day}: 9:00 AM, 2:00 PM, 4:00 PM'

manage_memory_tool = create_manage_memory_tool(namespace=('email_assistant', '{langgraph_user_id}', 'collection'))
search_memory_tool = create_search_memory_tool(namespace=('email_assistant', '{langgraph_user_id}', 'collection'))

agent_system_prompt_memory = '''
< Role > You are {full_name}'s executive assistant. </ Role >
< Tools > write_email, schedule_meeting, check_calendar_availability, manage_memory, search_memory </ Tools >
< Instructions > {instructions} </ Instructions >
'''
def create_prompt(state):
    return [{'role': 'system',
             'content': agent_system_prompt_memory.format(
                 instructions=prompt_instructions['agent_instructions'], **profile)}] + state['messages']

tools = [write_email, schedule_meeting, check_calendar_availability, manage_memory_tool, search_memory_tool]
response_agent = create_react_agent(get_chat_model(),
                                    tools=tools, prompt=create_prompt, store=store)

email_agent = StateGraph(State)
email_agent.add_node('triage_router', triage_router)
email_agent.add_node('response_agent', response_agent)
email_agent.add_edge(START, 'triage_router')
email_agent = email_agent.compile(store=store)

### The learning demo

Now the real test. A **vendor outreach** email arrives — phrased as a direct question. With no relevant
examples for this user yet, the agent falls back to its general rules and, because it reads like a direct
question, classifies it as **RESPOND**.

In [ ]:
email_input = {
    'author': 'Tom Jones <tom@apivendor.io>',
    'to': 'Youssef Hosni <Youssef.Hosni@ToDataBeyond.com>',
    'subject': 'Are you the right person for API tooling?',
    'email_thread': 'Hi Youssef, quick question - are you the right person on your team to talk to '
                    'about API monitoring tools? Happy to share how we help data science teams.',
}
_ = email_agent.invoke({'email_input': email_input},
                       config={'configurable': {'langgraph_user_id': 'Youssef'}})

In [ ]:
data = {'email': email_input, 'label': 'ignore'}
store.put(('email_assistant', 'Youssef', 'examples'), str(uuid.uuid4()), data)

# run the exact same email again
_ = email_agent.invoke({'email_input': email_input},
                       config={'configurable': {'langgraph_user_id': 'Youssef'}})

But suppose this user wants to **ignore** vendor outreach like this, even when it's dressed up as a
question. We teach the agent that preference by adding a few-shot example labelled `ignore`, scoped to
this user's `examples` namespace.

This time it's classified **IGNORE** — the agent searched its memory, found the example we just added, and learned this user's preference. Crucially, the memory is **user-specific**: a different user has no such example and reverts to the default.

In [ ]:
_ = email_agent.invoke({'email_input': email_input},
                       config={'configurable': {'langgraph_user_id': 'Jack'}})

For user "Jack", the same email is classified RESPOND again — he has no learned examples, so the
default behavior applies.

By adding **episodic memory** through few-shot examples, the assistant can now **learn from feedback**,
**personalize per user**, and **adapt over time** as more examples accumulate. This human-in-the-loop
learning is a cornerstone of robust agents. In the final LangMem chapter we add the last pillar —
**procedural memory** — so the agent can update its own instructions.